# Eksperimen Ekstraksi Fitur NLP (Ablation Study)

Notebook ini membandingkan beberapa kombinasi fitur NLP terhadap **baseline
time-series-only** yang sudah di-push tim di `src/baseline_kurs_tengah.ipynb`
(commit `4bcaee3`). Tidak ada file baseline maupun `src/preprocessing.ipynb`
yang diubah oleh notebook ini.

**Formulasi tugas** (mengikuti baseline, sudah dikonfirmasi tim):
- Target: klasifikasi biner **NAIK/TURUN** — apakah Kurs Tengah (rata-rata
  Kurs Jual & Kurs Beli) trading day berikutnya lebih tinggi dari hari ini.
- Split: **70/15/15 kronologis** (bukan random) — direproduksi persis dari
  baseline lewat `nlp_feature_extraction.load_baseline_df_model()` supaya
  batas train/val/test jatuh di baris yang identik.
- Model pembanding: **Logistic Regression** (sama seperti Model 2 di
  baseline), dengan `StandardScaler` yang di-`fit` hanya di train.
- Metrik: **Directional Accuracy** (`accuracy_score`) dan **F1-score**,
  dihitung dengan fungsi `evaluate_model` yang identik dengan baseline.

Kode reusable (loading, fitur, training, evaluasi) ada di
`src/nlp_feature_extraction.py` — file terpisah baru, tidak menimpa kode
siapapun.

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.append("../src")

import pandas as pd

from nlp_feature_extraction import (
    LEXICON_FEATURES,
    TS_FEATURES,
    TfidfSvdExtractor,
    add_lexicon_features,
    chronological_split,
    load_baseline_df_model,
    load_daily_news,
    train_eval_logreg,
)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

## 1. Rekonstruksi split baseline + gabungkan fitur NLP harian

`load_baseline_df_model` menjalankan ulang logika Kurs Tengah / Target
NAIK-TURUN / fitur lag & rolling / dropna persis seperti
`baseline_kurs_tengah.ipynb` cell 3-9, supaya jumlah baris dan urutan
tanggalnya identik. Fitur NLP harian (dari `data/processed/merge_data.csv`,
hasil `preprocessing.ipynb`) di-*left join* di atasnya berdasarkan
`Tanggal_dt`, lalu hari tanpa berita diisi 0 (netral) + ditandai `has_news`.

In [2]:
df_model = load_baseline_df_model(RAW_DIR)
news_daily = load_daily_news(PROCESSED_DIR)
news_daily = add_lexicon_features(news_daily)

full = df_model.merge(news_daily, on="Tanggal_dt", how="left")
assert len(full) == len(df_model), "jumlah baris berubah setelah merge -- cek key tanggal!"

full[LEXICON_FEATURES] = full[LEXICON_FEATURES].fillna(0.0)
full["all_text_lemmatized"] = full["all_text_lemmatized"].fillna("")

print(f"Total baris (setelah dropna lag/rolling, sama seperti baseline): {len(full)}")
print(f"Hari trading tanpa berita geopolitik: {(full['has_news'] == 0).sum()} dari {len(full)}")
full[["Tanggal_dt", "Kurs Tengah", "Target", "news_count", "has_news"]].head()

Total baris (setelah dropna lag/rolling, sama seperti baseline): 1196
Hari trading tanpa berita geopolitik: 138 dari 1196


,Tanggal_dt,Kurs Tengah,Target,news_count,has_news
0,2021-09-08,14195.005,1,1.0,1
1,2021-09-09,14266.000,1,1.0,1
2,2021-09-10,14272.000,0,1.0,1
3,2021-09-13,14225.005,1,4.0,1
4,2021-09-14,14260.000,0,2.0,1


In [3]:
train_df, val_df, test_df = chronological_split(full)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Periode train : {train_df['Tanggal_dt'].min().date()} s.d. {train_df['Tanggal_dt'].max().date()}")
print(f"Periode val   : {val_df['Tanggal_dt'].min().date()} s.d. {val_df['Tanggal_dt'].max().date()}")
print(f"Periode test  : {test_df['Tanggal_dt'].min().date()} s.d. {test_df['Tanggal_dt'].max().date()}")

Train: 837 | Val: 179 | Test: 180
Periode train : 2021-09-08 s.d. 2025-02-19
Periode val   : 2025-02-20 s.d. 2025-11-21
Periode test  : 2025-11-24 s.d. 2026-08-31


> **Catatan reproduksibilitas:** jumlah baris (837/179/180) dan periode
> train (2021-09-08 s.d. 2025-02-19) identik dengan yang tercatat di
> `baseline_kurs_tengah.ipynb`. Tanggal akhir periode test di sini berselisih
> 1 hari (berakhir di trading day sebelum baris terakhir file kurs, karena
> baris paling akhir selalu dibuang -- tidak punya target hari berikutnya)
> dibanding output tersimpan di notebook baseline; itu wajar karena kedua
> notebook dijalankan pada waktu yang berbeda, bukan indikasi split yang
> berbeda. Yang menentukan keadilan perbandingan adalah **jumlah baris &
> rasio split yang sama**, dan itu sudah dikonfirmasi identik.

## 2. Baseline (dari `src/baseline_kurs_tengah.ipynb`, tidak dijalankan ulang di sini)

Dicatat manual dari output baseline supaya bisa masuk ke satu tabel
perbandingan yang sama.

In [4]:
baseline_results = [
    {"model": "Naive (Majority Class)", "dataset": "Val", "directional_accuracy": 0.525140, "f1_score": 0.688645},
    {"model": "Naive (Majority Class)", "dataset": "Test", "directional_accuracy": 0.577778, "f1_score": 0.732394},
    {"model": "Naive (Persistence Arah)", "dataset": "Val", "directional_accuracy": 0.513966, "f1_score": 0.539683},
    {"model": "Naive (Persistence Arah)", "dataset": "Test", "directional_accuracy": 0.550000, "f1_score": 0.608696},
    {"model": "Baseline: Logistic Regression (TS-only)", "dataset": "Val", "directional_accuracy": 0.547486, "f1_score": 0.649351},
    {"model": "Baseline: Logistic Regression (TS-only)", "dataset": "Test", "directional_accuracy": 0.544444, "f1_score": 0.549451},
]
results = list(baseline_results)

## 3. Skenario A — Lexicon-only (VADER + Loughran-McDonald + GPR index)

Dimensi rendah (5 kolom): `vader_compound`, `lm_polarity`,
`lm_subjectivity`, `gpr_density`, `has_news`. Tidak butuh fitting dari data
(semua berbasis kamus/aturan tetap), jadi aman dihitung sebelum split.

In [5]:
def record(rows, dataset_from_name=True):
    for r in rows:
        name, ds = r["model"].rsplit(" - ", 1)
        results.append({"model": name, "dataset": ds, "directional_accuracy": r["directional_accuracy"], "f1_score": r["f1_score"]})

scenario_a_cols = TS_FEATURES + LEXICON_FEATURES
rows_a, _ = train_eval_logreg(scenario_a_cols, train_df, val_df, test_df, "Skenario A: TS + Lexicon")
record(rows_a)
pd.DataFrame(rows_a)

,model,directional_accuracy,f1_score
0,Skenario A: TS + Lexicon - Val,0.497207,0.608696
1,Skenario A: TS + Lexicon - Test,0.527778,0.508671


## 4. Skenario B — Skenario A + TF-IDF/SVD (3 komponen)

`TfidfSvdExtractor` di-`fit` **hanya pada `all_text_lemmatized` milik
`train_df`**, lalu `.transform()` saja untuk val/test -- mencegah data
leakage vocabulary dari masa depan.

In [6]:
extractor_b = TfidfSvdExtractor(n_components=3, min_df=5)
extractor_b.fit(train_df["all_text_lemmatized"])
print(f"Ukuran vocabulary (fit di train saja): {len(extractor_b._vectorizer.vocabulary_)}")
print(f"Explained variance ratio (3 komponen): {extractor_b.explained_variance_ratio_.sum():.3f}")

svd_train_b = extractor_b.transform(train_df["all_text_lemmatized"])
svd_val_b = extractor_b.transform(val_df["all_text_lemmatized"])
svd_test_b = extractor_b.transform(test_df["all_text_lemmatized"])

train_b = pd.concat([train_df.reset_index(drop=True), svd_train_b.reset_index(drop=True)], axis=1)
val_b = pd.concat([val_df.reset_index(drop=True), svd_val_b.reset_index(drop=True)], axis=1)
test_b = pd.concat([test_df.reset_index(drop=True), svd_test_b.reset_index(drop=True)], axis=1)

scenario_b_cols = scenario_a_cols + list(svd_train_b.columns)
rows_b, _ = train_eval_logreg(scenario_b_cols, train_b, val_b, test_b, "Skenario B: TS + Lexicon + TF-IDF(3)")
record(rows_b)
pd.DataFrame(rows_b)

Ukuran vocabulary (fit di train saja): 5000
Explained variance ratio (3 komponen): 0.079


,model,directional_accuracy,f1_score
0,Skenario B: TS + Lexicon + TF-IDF(3) - Val,0.480447,0.56338
1,Skenario B: TS + Lexicon + TF-IDF(3) - Test,0.511111,0.50000


## 5. Skenario C — Full TF-IDF/SVD (20 komponen)

Komponen SVD diperbanyak untuk melihat apakah performa membaik atau malah
overfitting dibanding Skenario B.

In [7]:
extractor_c = TfidfSvdExtractor(n_components=20, min_df=5)
extractor_c.fit(train_df["all_text_lemmatized"])
print(f"Explained variance ratio (20 komponen): {extractor_c.explained_variance_ratio_.sum():.3f}")

svd_train_c = extractor_c.transform(train_df["all_text_lemmatized"])
svd_val_c = extractor_c.transform(val_df["all_text_lemmatized"])
svd_test_c = extractor_c.transform(test_df["all_text_lemmatized"])

train_c = pd.concat([train_df.reset_index(drop=True), svd_train_c.reset_index(drop=True)], axis=1)
val_c = pd.concat([val_df.reset_index(drop=True), svd_val_c.reset_index(drop=True)], axis=1)
test_c = pd.concat([test_df.reset_index(drop=True), svd_test_c.reset_index(drop=True)], axis=1)

scenario_c_cols = scenario_a_cols + list(svd_train_c.columns)
rows_c, _ = train_eval_logreg(scenario_c_cols, train_c, val_c, test_c, "Skenario C: TS + Lexicon + TF-IDF(20)")
record(rows_c)
pd.DataFrame(rows_c)

Explained variance ratio (20 komponen): 0.192


,model,directional_accuracy,f1_score
0,Skenario C: TS + Lexicon + TF-IDF(20) - Val,0.525140,0.535519
1,Skenario C: TS + Lexicon + TF-IDF(20) - Test,0.488889,0.386667


## 6. Tabel Perbandingan Akhir

In [8]:
summary = pd.DataFrame(results)
pivot = summary.pivot(index="model", columns="dataset", values="directional_accuracy")
pivot["f1_test"] = summary[summary["dataset"] == "Test"].set_index("model")["f1_score"]
pivot["f1_val"] = summary[summary["dataset"] == "Val"].set_index("model")["f1_score"]
pivot = pivot[["Val", "Test", "f1_val", "f1_test"]].sort_values("Test", ascending=False)

best_baseline_test = baseline_results_test_max = max(
    r["directional_accuracy"] for r in baseline_results if r["dataset"] == "Test"
)
pivot["selisih_vs_best_baseline_test"] = pivot["Test"] - best_baseline_test

output_path = Path("nlp_experiment_results.csv")
pivot.to_csv(output_path)
print(f"Tabel hasil disimpan ke: {output_path.resolve()}")
pivot

Tabel hasil disimpan ke: /Users/farsyaant/nlpproject-grestercinta/notebook/nlp_experiment_results.csv


dataset,Val,Test,f1_val,f1_test,selisih_vs_best_baseline_test
model,,,,,
Naive (Majority Class),0.525140,0.577778,0.688645,0.732394,0.000000
Naive (Persistence Arah),0.513966,0.550000,0.539683,0.608696,-0.027778
Baseline: Logistic Regression (TS-only),0.547486,0.544444,0.649351,0.549451,-0.033334
Skenario A: TS + Lexicon,0.497207,0.527778,0.608696,0.508671,-0.050000
Skenario B: TS + Lexicon + TF-IDF(3),0.480447,0.511111,0.563380,0.500000,-0.066667
Skenario C: TS + Lexicon + TF-IDF(20),0.525140,0.488889,0.535519,0.386667,-0.088889


## 7. Insight

- **Tidak ada skenario fitur NLP (A/B/C) yang mengalahkan baseline terbaik**
  (Naive Majority Class, *directional accuracy* test 0.578). Skenario A
  (lexicon-only) paling dekat (0.528) tapi masih di bawah baseline
  Logistic Regression time-series-only (0.544) dan kedua naive baseline.
- **Menambah TF-IDF/SVD (Skenario B, C) tidak membantu dan cenderung
  menurunkan performa test** dibanding Skenario A lexicon-only (0.511 dan
  0.489 vs 0.528). *Explained variance ratio* TF-IDF+SVD juga rendah (8%
  untuk 3 komponen, 19% untuk 20 komponen) — sinyal dari teks berita harian
  yang diagregasi (gabungan artikel per hari, lalu direduksi jadi beberapa
  komponen SVD) tampaknya terlalu lemah/ternoise untuk model linear
  sederhana ini.
- **Dugaan penyebab:** (1) pergerakan arah kurs harian sangat dekat
  random-walk secara historis, jadi bar akurasi 55-58% dari naive baseline
  sudah cukup tinggi untuk dikalahkan; (2) distribusi target timpang
  (~55% NAIK), yang menguntungkan model "malas" (majority-class-like)
  ketimbang model yang benar-benar coba membaca sinyal berita; (3) fitur
  NLP dihitung di level agregasi harian (gabungan seluruh artikel per hari
  jadi satu blob teks), bukan per-artikel lalu dirata-rata — kemungkinan
  menghilangkan nuansa/menambah noise dibanding pendekatan per-artikel yang
  dipakai tim di `preprocessing.ipynb`.
- **Rekomendasi lanjutan** (belum dikerjakan di notebook ini): (a) coba
  hitung fitur sentimen di level artikel dulu baru diagregasi (mean/sum
  per hari) seperti pendekatan `preprocessing.ipynb`, alih-alih
  menggabungkan teks jadi satu blob per hari; (b) uji fitur NLP dengan lag
  1-3 hari (sentimen hari-hari sebelumnya, bukan hanya hari yang sama);
  (c) pertimbangkan target volatilitas/multi-class alih-alih biner
  NAIK/TURUN yang secara historis sangat sulit diprediksi untuk pasangan
  mata uang likuid seperti USD/IDR.